In [1]:
!poetry install -q

In [2]:
%load_ext autoreload
%autoreload 2

import os
import sys
import datetime
import numpy as np
import pandas as pd
from dotenv import load_dotenv

# OpenMP 다중 로드 허용 및 스레드 경쟁 방지 환경 변수 (최상단 주입 필수)
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["VECLIB_MAXIMUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"

# 1. 프로젝트 경로 설정 및 환경 변수 명시적 로드
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.append(project_root)

# Docker 네트워크 외부(Host OS)에서 실행되는 Jupyter를 위한 DNS 해석 우회 처리
local_s3_endpoint = os.environ.get("LOCAL_S3_ENDPOINT", "")
if "localstack" in local_s3_endpoint:
    os.environ["LOCAL_S3_ENDPOINT"] = local_s3_endpoint.replace("localstack", "localhost")

In [3]:
# DataFrame 출력 생략 방지 옵션 설정
pd.set_option('display.max_columns', None)        # 숨김 없이 모든 컬럼 출력
pd.set_option('display.max_colwidth', None)       # 컬럼 안의 긴 텍스트(Dict/List) 전체 출력
pd.set_option('display.expand_frame_repr', False) # 가로 너비 초과 시 줄바꿈 방지
pd.set_option('display.max_rows', 50)             # 필요시 최대 출력 행 수 조정

In [4]:
# ==============================================================================
# [셀 0] 주피터 분석 환경 전용 MinIO 및 MLflow 환경 변수 세팅
# ==============================================================================
import os
import warnings
import logging
from dotenv import find_dotenv, load_dotenv
from mlflow.tracking import MlflowClient
from pandas.errors import PerformanceWarning
from src.model.tracker.mlflow_tracker import MLflowTracker

# 1. 루트 디렉터리의 .env 파일 자동 탐색 및 로드
load_dotenv(find_dotenv())

# 2. MinIO S3 및 AWS 자격 증명 환경 변수 세팅
os.environ["AWS_ACCESS_KEY_ID"] = os.getenv("AWS_ACCESS_KEY_ID")
os.environ["AWS_SECRET_ACCESS_KEY"] = os.getenv("AWS_SECRET_ACCESS_KEY")
os.environ["AWS_DEFAULT_REGION"] = os.getenv("AWS_DEFAULT_REGION")
os.environ["LOCAL_S3_ENDPOINT"] = os.getenv("MLFLOW_S3_ENDPOINT_URL")

# 3. MLflow Tracking & S3 아티팩트 스토어 연동 설정
MLFLOW_TRACKING_URI = os.getenv("MLFLOW_TRACKING_URI")
os.environ["MLFLOW_TRACKING_URI"] = MLFLOW_TRACKING_URI
os.environ["MLFLOW_S3_ENDPOINT_URL"] = os.getenv("MLFLOW_S3_ENDPOINT_URL")
os.environ["MLFLOW_S3_IGNORE_TLS"] = "true"
# MLflow 공식 표준 출력 URL 링크 차단 환경변수 (소프트웨어 공식 플래그)
os.environ["MLFLOW_SUPPRESS_PRINTING_URL_TO_STDOUT"] = "true"

# 4. 경고 필터링
warnings.filterwarnings("ignore", category=PerformanceWarning)
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=UserWarning)

# MLFlow 활성화
EXPERIMENT_NAME: str = "Champion_Dataset_Selection"
tracker = MLflowTracker(experiment_name=EXPERIMENT_NAME)

/Users/junsu/code/Project/AssetMind/apps/data-pipeline/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


🎯 [MLflow Tracker Active] Experiment: 'Champion_Dataset_Selection' (ID: 2)


In [5]:
# ==============================================================================
# [셀 1] 골드 레이어 18종 파생 데이터셋 고속 로드 및 실시간 스트리밍 관제
# ==============================================================================
from notebooks.utils.get_best_dataset.loader import GoldDatasetLoader

# ------------------------------------------------------------------------------
# Step 1. GoldDatasetLoader 인스턴스화 및 18종 골드 데이터셋 일괄 수집/정제
# ------------------------------------------------------------------------------
gold_dataset_loader = GoldDatasetLoader()
gold_dataset_repository, ingestion_summary_dataframe = gold_dataset_loader.load_all_datasets()

# ------------------------------------------------------------------------------
# Step 2. 전체 수집 정산 대시보드 사출
# ------------------------------------------------------------------------------
display(ingestion_summary_dataframe)


 📊 [Total Dataset Ingestion Summary Dashboard] Executed in 2027.01s


,Dataset Bucket ID,Total Rows,Total Features,Start Date,End Date,Load Time
0,bucket_impute_locf_detect_iqr_refine_clipping,3877,1456,2016-01-01,2026-08-23,117.09s
1,bucket_impute_locf_detect_iqr_refine_masking,3877,1456,2016-01-01,2026-08-23,108.92s
2,bucket_impute_locf_detect_isolation_forest_refine_clipping,3877,1456,2016-01-01,2026-08-23,110.37s
3,bucket_impute_locf_detect_isolation_forest_refine_masking,3877,1456,2016-01-01,2026-08-23,112.82s
4,bucket_impute_locf_detect_zscore_refine_clipping,3877,1456,2016-01-01,2026-08-23,113.01s
5,bucket_impute_locf_detect_zscore_refine_masking,3877,1456,2016-01-01,2026-08-23,110.93s
6,bucket_impute_log_return_detect_iqr_refine_clipping,3877,1456,2016-01-01,2026-08-23,109.22s
7,bucket_impute_log_return_detect_iqr_refine_masking,3877,1456,2016-01-01,2026-08-23,110.15s
8,bucket_impute_log_return_detect_isolation_forest_refine_clipping,3877,1456,2016-01-01,2026-08-23,108.32s
9,bucket_impute_log_return_detect_isolation_forest_refine_masking,3877,1456,2016-01-01,2026-08-23,108.61s


In [6]:
# [임시 테스트용] 데이터 슬라이싱 (추후 전체 백필 완료 시 주석 처리)
TEST_CUTOFF_DATE = "2026-08-21"
gold_dataset_repository = {
    bucket_id: df.loc[:TEST_CUTOFF_DATE].copy()
    for bucket_id, df in gold_dataset_repository.items()
}
print(f"✂️ [Test Mode Active] 18종 데이터셋을 {TEST_CUTOFF_DATE} 기준으로 슬라이싱 완료")

✂️ [Test Mode Active] 18종 데이터셋을 2026-08-21 기준으로 슬라이싱 완료


In [7]:
# ==============================================================================
# [셀 2] 18종 데이터셋 대상 파생 피처 생성 및 시계열 타겟 변수(T+20) 일괄 사출
# ==============================================================================
from src.feature.feature_service import FeatureService
from notebooks.utils.get_best_dataset.feature import (
    generate_domain_features,
    report_domain_groups
)

# ------------------------------------------------------------------------------
# Step 1. 18종 골드 데이터셋 일괄 파생 피처 엔지니어링 집행
# ------------------------------------------------------------------------------
feature_service = FeatureService()
engineered_dataset_repository, feature_summary_dataframe = generate_domain_features(
    datasets=gold_dataset_repository,
    feature_service=feature_service
)
display(feature_summary_dataframe)

# ------------------------------------------------------------------------------
# Step 2. 6대 도메인 피처 그룹 규격 및 사출 무결성 정밀 감사표 (Audit Report)
# ------------------------------------------------------------------------------
sample_bucket_id = next(iter(gold_dataset_repository.keys()))
domain_group_report = report_domain_groups(
    raw_sample=gold_dataset_repository[sample_bucket_id],
    engineered_sample=engineered_dataset_repository[sample_bucket_id]
)

print("\n" + "=" * 122)
print(" 🔬 [Feature Engineering Domain Group Specification Report]")
print("=" * 122)
display(domain_group_report)

/Users/junsu/code/Project/AssetMind/apps/data-pipeline/src/feature/tasks/target_feature.py:1: SyntaxWarning: invalid escape sequence '\l'
  """
/Users/junsu/code/Project/AssetMind/apps/data-pipeline/src/feature/tasks/trend_momentum.py:1: SyntaxWarning: invalid escape sequence '\l'
  """
/Users/junsu/code/Project/AssetMind/apps/data-pipeline/src/feature/tasks/volatility_risk.py:1: SyntaxWarning: invalid escape sequence '\s'
  """


 🚀 [Batch Feature Engineering Launch] Target Datasets: 18 Sets


Engineering Features: 100%|██████████| 18/18 [00:15<00:00,  1.14dataset/s, Processing: bucket_impute_moving_average_detect...]


 📊 [Feature Engineering Report] Completed in 15.83s


,Dataset Bucket ID,Input Shape,Output Shape,Generated Features,Target Status,Elapsed Time
0,bucket_impute_locf_detect_iqr_refine_clipping,"(3875, 1456)","(3875, 3382)",+1926,VALID (target_return_20d),0.983s
1,bucket_impute_locf_detect_iqr_refine_masking,"(3875, 1456)","(3875, 3382)",+1926,VALID (target_return_20d),0.914s
2,bucket_impute_locf_detect_isolation_forest_refine_clipping,"(3875, 1456)","(3875, 3382)",+1926,VALID (target_return_20d),0.836s
3,bucket_impute_locf_detect_isolation_forest_refine_masking,"(3875, 1456)","(3875, 3382)",+1926,VALID (target_return_20d),0.924s
4,bucket_impute_locf_detect_zscore_refine_clipping,"(3875, 1456)","(3875, 3382)",+1926,VALID (target_return_20d),0.859s
5,bucket_impute_locf_detect_zscore_refine_masking,"(3875, 1456)","(3875, 3382)",+1926,VALID (target_return_20d),0.865s
6,bucket_impute_log_return_detect_iqr_refine_clipping,"(3875, 1456)","(3875, 3382)",+1926,VALID (target_return_20d),0.846s
7,bucket_impute_log_return_detect_iqr_refine_masking,"(3875, 1456)","(3875, 3382)",+1926,VALID (target_return_20d),0.932s
8,bucket_impute_log_return_detect_isolation_forest_refine_clipping,"(3875, 1456)","(3875, 3382)",+1926,VALID (target_return_20d),0.861s
9,bucket_impute_log_return_detect_isolation_forest_refine_masking,"(3875, 1456)","(3875, 3382)",+1926,VALID (target_return_20d),0.869s



 🔬 [Feature Engineering Domain Group Specification Report]


,피처 그룹 (Feature Group),활성화 상태,표준 팩터 규격,총 사출 피처 수,표준 파생 팩터 목록
0,Target Feature,ENABLED,1 / 1 종,1 개,target_return_20d
1,Trend & Momentum (Multi-Asset),ENABLED,7 / 7 종,959 개,"return_lag_5d, return_lag_20d, return_lag_60d, return_lag_120d, ma_ratio_5_20, ma_ratio_20_60, risk_adjusted_return_20d"
2,Volatility & Risk (Multi-Asset),ENABLED,7 / 7 종,959 개,"volatility_20d, volatility_60d, vol_regime_ratio, rolling_skew_20d, rolling_kurt_20d, price_position_20d, norm_atr_20d"
3,Macro & Cross-Asset,DISABLED,0 / 4 종,0 개,-
4,Derivatives & Volume,ENABLED,3 / 4 종,3 개,"proxy_basis_rate, futures_intraday_range, volume_anomaly_20d"
5,Calendar & Seasonality,ENABLED,4 / 4 종,4 개,"month_sin, month_cos, is_month_end, is_quarter_end"


In [8]:
# ==============================================================================
# [셀 3] 18종 데이터셋 대상 Data Leakage 방지 시계열 일괄 분할 (Train / Test with Purged Gap)
# ==============================================================================
from src.model.dataset.splitter import DatasetSplitter
from notebooks.utils.get_best_dataset.splitter import batch_split_datasets, report_feature_leakage

# ------------------------------------------------------------------------------
# Step 1. DatasetSplitter 초기화 및 18종 데이터셋 일괄 시계열 분할
# ------------------------------------------------------------------------------
raw_columns_to_exclude = list(next(iter(gold_dataset_repository.values())).columns)

dataset_splitter = DatasetSplitter(
    split_ratios=(0.8, 0.2),
    forecast_horizon=20,
    exclude_features=raw_columns_to_exclude
)
split_dataset_repository = batch_split_datasets(
    datasets=engineered_dataset_repository,
    splitter=dataset_splitter
)

# ------------------------------------------------------------------------------
# Step 2. Data Leakage 전수 검증 및 분할 정산 리포트 사출
# ------------------------------------------------------------------------------
split_summary_report = report_feature_leakage(
    split_repository=split_dataset_repository,
    raw_columns=raw_columns_to_exclude
)
display(split_summary_report)

# ------------------------------------------------------------------------------
# Step 3. 대표 데이터셋 시계열 구간 및 Purged Gap 무결성 상세 리포트 사출
# ------------------------------------------------------------------------------
sample_bucket_id = next(iter(split_dataset_repository.keys()))
display(dataset_splitter.summarize(split_datasets=split_dataset_repository[sample_bucket_id]))

 🚀 [Batch Time-Series Splitting Launch] Mode: Train/Test (80:20) | Gap: 20d | Target: 18 Sets


,Dataset Bucket ID,Target Exclusion,X_train Shape,y_train Shape,purged_gap Shape,X_test Shape,y_test Shape,X_inference Shape
0,bucket_impute_locf_detect_iqr_refine_clipping,ALL RAW DROPPED,"(3068, 1925)","(3068,)","(20, 1925)","(767, 1925)","(767,)","(20, 1925)"
1,bucket_impute_locf_detect_iqr_refine_masking,ALL RAW DROPPED,"(3068, 1925)","(3068,)","(20, 1925)","(767, 1925)","(767,)","(20, 1925)"
2,bucket_impute_locf_detect_isolation_forest_refine_clipping,ALL RAW DROPPED,"(3068, 1925)","(3068,)","(20, 1925)","(767, 1925)","(767,)","(20, 1925)"
3,bucket_impute_locf_detect_isolation_forest_refine_masking,ALL RAW DROPPED,"(3068, 1925)","(3068,)","(20, 1925)","(767, 1925)","(767,)","(20, 1925)"
4,bucket_impute_locf_detect_zscore_refine_clipping,ALL RAW DROPPED,"(3068, 1925)","(3068,)","(20, 1925)","(767, 1925)","(767,)","(20, 1925)"
5,bucket_impute_locf_detect_zscore_refine_masking,ALL RAW DROPPED,"(3068, 1925)","(3068,)","(20, 1925)","(767, 1925)","(767,)","(20, 1925)"
6,bucket_impute_log_return_detect_iqr_refine_clipping,ALL RAW DROPPED,"(3047, 1925)","(3047,)","(20, 1925)","(762, 1925)","(762,)","(20, 1925)"
7,bucket_impute_log_return_detect_iqr_refine_masking,ALL RAW DROPPED,"(3047, 1925)","(3047,)","(20, 1925)","(762, 1925)","(762,)","(20, 1925)"
8,bucket_impute_log_return_detect_isolation_forest_refine_clipping,ALL RAW DROPPED,"(3047, 1925)","(3047,)","(20, 1925)","(762, 1925)","(762,)","(20, 1925)"
9,bucket_impute_log_return_detect_isolation_forest_refine_masking,ALL RAW DROPPED,"(3047, 1925)","(3047,)","(20, 1925)","(762, 1925)","(762,)","(20, 1925)"


,Partition,Date Range,Shape,Role
0,X_train,2016-01-01 ~ 2024-06-03,"(3,068, 1,925)",모델 가중치 학습용 피처 세트
1,y_train,2016-01-01 ~ 2024-06-03,"3,068",모델 가중치 학습용 타겟 벡터
2,purged_gap,2024-06-04 ~ 2024-06-23,"(20, 1,925)",데이터 누수 방지용 삭제 구간 (Purged Gap)
3,X_test,2024-06-24 ~ 2026-08-01,"(767, 1,925)",최종 검증(Out-of-Sample) 피처 세트
4,y_test,2024-06-24 ~ 2026-08-01,767,최종 검증(Out-of-Sample) 타겟 벡터
5,X_inference,2026-08-02 ~ 2026-08-21,"(20, 1,925)",실시간 추론 및 페이퍼 트레이딩 피처 세트 (y 결손)


In [9]:
# ==============================================================================
# [셀 4] 18종 데이터셋 피처 셀렉션 (Noise & Collinearity Filter ➔ Robust Scaling ➔ 2-Pillar Selection)
# ==============================================================================
import pandas as pd
from notebooks.utils.get_best_dataset.selector import batch_select_features

# ------------------------------------------------------------------------------
# Step 1. 피처 엔지니어링 후행 전처리 하이퍼파라미터 상수 정의
# ------------------------------------------------------------------------------
MAX_MISSING_RATIO: float = 0.2
MAX_ZERO_RATIO: float = 0.8
PEARSON_THRESHOLD: float = 0.85
DISTANCE_THRESHOLD: float = 0.40
TARGET_CUMULATIVE_THRESHOLD: float = 0.95
MIN_FEATURES_BOUND: int = 10
MAX_FEATURES_BOUND: int = 100

# ------------------------------------------------------------------------------
# Step 2. 18종 데이터셋 일괄 5단계 후행 전처리 및 동적 피처 선별 집행
# ------------------------------------------------------------------------------
model_ready_repository, selection_summary_report, representative_audit_report = batch_select_features(
    split_repository=split_dataset_repository,
    max_missing_ratio=MAX_MISSING_RATIO,
    max_zero_ratio=MAX_ZERO_RATIO,
    pearson_threshold=PEARSON_THRESHOLD,
    distance_threshold=DISTANCE_THRESHOLD,
    cumulative_threshold=TARGET_CUMULATIVE_THRESHOLD,
    min_features=MIN_FEATURES_BOUND,
    max_features=MAX_FEATURES_BOUND
)

# ------------------------------------------------------------------------------
# Step 3. 18개 데이터셋 피처 셀렉션 종합 정산 대시보드 사출
# ------------------------------------------------------------------------------
display(selection_summary_report)

# ------------------------------------------------------------------------------
# Step 4. 대표 데이터셋 5단계 후행 전처리 세부 감사표 (Audit Table) 사출
# ------------------------------------------------------------------------------
sample_dataset_id = next(iter(split_dataset_repository.keys()))
print(f"\n🔬 [Representative Dataset Feature Selection Audit: '{sample_dataset_id}']")
display(representative_audit_report)

 🚀 [Batch Feature Selection Launch] Target Datasets: 18 Sets | 2-Pillar Hybrid + Robust Scaler


🔍 [2-Pillar Selection]: 100%|██████████| 18/18 [11:48<00:00, 39.34s/dataset, Done: bucket_impute_moving_aver... (18.9s, Feats: 12)]

 ✅ [Batch Feature Selection Completed] Total Elapsed Time: 708.17s


,Dataset Bucket ID,Initial Features,After Noise Filter,After Collinear Filter,Selected Features,Cumulative Coverage,Elapsed Time
0,bucket_impute_locf_detect_iqr_refine_clipping,1925,1883,608,59,95.0%,40.58s
1,bucket_impute_locf_detect_iqr_refine_masking,1925,1863,798,57,95.1%,48.29s
2,bucket_impute_locf_detect_isolation_forest_refine_clipping,1925,1883,608,59,95.0%,39.77s
3,bucket_impute_locf_detect_isolation_forest_refine_masking,1925,1863,490,10,95.6%,18.50s
4,bucket_impute_locf_detect_zscore_refine_clipping,1925,1883,608,59,95.0%,39.20s
5,bucket_impute_locf_detect_zscore_refine_masking,1925,1863,493,15,95.2%,18.47s
6,bucket_impute_log_return_detect_iqr_refine_clipping,1925,1882,578,84,95.1%,47.41s
7,bucket_impute_log_return_detect_iqr_refine_masking,1925,1862,849,79,95.1%,62.07s
8,bucket_impute_log_return_detect_isolation_forest_refine_clipping,1925,1882,578,84,95.1%,47.17s
9,bucket_impute_log_return_detect_isolation_forest_refine_masking,1925,1867,500,30,95.1%,18.72s



🔬 [Representative Dataset Feature Selection Audit: 'bucket_impute_locf_detect_iqr_refine_clipping']


,잔여 피처 수 (Features),변동 내역 (Changes),적용 기준 (Fit Strategy)
단계 (Pipeline Step),,,
1. Noise Filter (Constant & Missing & Zero),"1,883","-42 Features (Constant: 0, Missing: 15, Zero: 27)","상수 및 결측률(> 20%), 0값(≥ 80%)"
2. Pearson Collinearity,979,-904 Features,피어슨 선형 상관계수(|r| ≥ 0.85)
3. Hierarchical Feature Clustering (HFC),608,-371 Features,상관거리 계층 군집화 (Distance ≤ 0.4)
4. Robust Feature Scaling,608,No Dimension Change (All Partitions Normalized),학습 데이터 중앙값 및 IQR 통계량 기준
5. Dynamic 2-Pillar Selection (Top 59),59,Selected 59 Features,ElasticNet + LightGBM (Target: 95%)


In [10]:
# ==============================================================================
# [셀 5] 18종 데이터셋 × 3대 모델 스크리닝 및 챔피언 데이터셋(champion_dataset.pkl) 확정
# ==============================================================================
from notebooks.utils.get_best_dataset.screener import run_batch_screening, save_champion_dataset

# ------------------------------------------------------------------------------
# Step 1. 18종 데이터셋 × 3대 베이스라인 모델 일괄 스크리닝 및 종합 랭킹 수립
# ------------------------------------------------------------------------------
raw_ranking_report, champion_dataset_id, champion_meta, display_ranking_report = run_batch_screening(
    model_ready_repository=model_ready_repository
)

# ------------------------------------------------------------------------------
# Step 2. 완성형 1위 Champion Dataset 파티션 직렬화 저장 및 무결성 검증
# ------------------------------------------------------------------------------
artifact_meta = save_champion_dataset(
    model_ready_repository=model_ready_repository,
    champion_dataset_id=champion_dataset_id,
    output_artifact_path="champion_dataset.pkl"
)

# ------------------------------------------------------------------------------
# Step 3. 챔피언 확정 배너 및 18종 스크리닝 종합 정산 대시보드 사출
# ------------------------------------------------------------------------------
print("\n" + "=" * 122)
print(f" 🏆 [Champion Dataset Selected] '{champion_dataset_id}' 확정!")
print(f" 📌 Validation 성능: Avg MDA {champion_meta['avg_mda'] * 100:.2f}% | Avg RMSE {champion_meta['avg_rmse']:.6f} | Selected Features: {champion_meta['features']}개")
print(f" 💾 완성형 데이터셋 파티션 저장 완료: '{artifact_meta['artifact_path']}' (크기: {artifact_meta['file_size_kb']:.2f} KB | Train: {artifact_meta['train_shape']} | Test: {artifact_meta['test_shape']})")
print(f" ⏱️ 총 소요 시간: {champion_meta['total_elapsed']:.2f}s")
print("=" * 122)

display(display_ranking_report)

 🚀 [Batch Dataset Screening Launch] Target Datasets: 18 Sets × 3 Models (54 Evaluations)


🔍 [Dataset Screening]: 100%|██████████| 18/18 [00:03<00:00,  5.44set/s]

 ✅ [Screening Completed] Total Elapsed Time: 3.31s

 🏆 [Champion Dataset Selected] 'bucket_impute_moving_average_detect_zscore_refine_masking' 확정!
 📌 Validation 성능: Avg MDA 71.80% | Avg RMSE 0.296074 | Selected Features: 12개
 💾 완성형 데이터셋 파티션 저장 완료: 'champion_dataset.pkl' (크기: 893.88 KB | Train: (3068, 12) | Test: (767, 12))
 ⏱️ 총 소요 시간: 3.31s


,Selected Features,Avg MDA,Avg RMSE,Avg MAE,Best Model,Best Model MDA,Best Model RMSE,Dataset Bucket ID,Elapsed Time
Rank,,,,,,,,,
1,12,71.80%,0.296074,0.212229,XGBoostRegressor,74.15%,0.262964,bucket_impute_moving_average_detect_zscore_refine_masking,0.11s
2,30,71.62%,0.398617,0.291074,ElasticNet,74.90%,0.390721,bucket_impute_log_return_detect_isolation_forest_refine_masking,0.15s
3,14,71.11%,0.311612,0.229947,XGBoostRegressor,73.89%,0.291392,bucket_impute_moving_average_detect_isolation_forest_refine_masking,0.12s
4,25,70.92%,0.420745,0.313940,ElasticNet,74.90%,0.391164,bucket_impute_log_return_detect_zscore_refine_masking,0.15s
5,15,70.02%,0.310244,0.221000,XGBoostRegressor,70.10%,0.259621,bucket_impute_locf_detect_zscore_refine_masking,0.12s
6,10,67.28%,0.346216,0.265238,ElasticNet,69.84%,0.377073,bucket_impute_locf_detect_isolation_forest_refine_masking,0.11s
7,84,66.93%,0.235733,0.152570,RandomForestRegressor,68.86%,0.192700,bucket_impute_log_return_detect_iqr_refine_clipping,0.28s
8,84,66.93%,0.235733,0.152570,RandomForestRegressor,68.86%,0.192700,bucket_impute_log_return_detect_isolation_forest_refine_clipping,0.28s
9,84,66.93%,0.235733,0.152570,RandomForestRegressor,68.86%,0.192700,bucket_impute_log_return_detect_zscore_refine_clipping,0.28s
